In [ ]:
# Install the required packages:
# - langchain: core framework
# - langchain-openai: OpenAI model integration
!pip install -q langchain langchain-openai


In [ ]:
# Standard imports for working with LangChain and OpenAI
from google.colab import userdata
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from pydantic import SecretStr
from typing import List

# Safely load the OpenAI API key from Colab secrets
openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# Helper to print all messages in a conversation in a human-readable way
def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()


In [ ]:
# Define two simple tools to demonstrate different tool_choice behaviors.
# The @tool decorator registers these functions so the model can call them.

@tool
def do_nothing() -> str:
    """
    This tool does absolutely nothing. Do not call it!
    """
    # This tool is intentionally useless — it's used to show "forced" calling
    # where the model is made to call it even when it shouldn't logically need to.
    return "Nothing..."

@tool
def get_interesting_fact() -> str:
    """
    This tool will discover an interesting fact to you.
    """
    # Returns a static interesting fact — in a real app this could call an API
    return "The Earth is actually not a perfect sphere."


In [ ]:
# Collect all tools into a list and build a registry (name → tool) for easy lookup.
# The registry lets us call the correct tool by name when the model requests one.
tools = [do_nothing, get_interesting_fact]
tools_registry = { t.name: t for t in tools }


## "Auto" tool choice

The model decides freely if / when / what tool to call.

In [ ]:
# --- "auto" tool_choice ---
# With tool_choice="auto" the model freely decides whether to call a tool or not.
# In this case, the model receives a friendly greeting and will likely reply without calling any tool.
# The default tool_choice is "auto"
openai_auto_tool_choice_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key).bind_tools(tools, tool_choice="auto")

auto_tool_choice_conversation = [HumanMessage("Hello! I am a software developer. What about you?")]

# Invoke the model — it will decide on its own whether to use a tool
auto_tool_choice_response = openai_auto_tool_choice_model.invoke(auto_tool_choice_conversation)

# Append the model's response to the conversation list to keep the full history
auto_tool_choice_conversation.append(auto_tool_choice_response)


In [ ]:
# Print the auto tool_choice conversation — expect a direct text reply with no tool call
print_conversation(auto_tool_choice_conversation)


## "Any" tool choice

In [ ]:
# --- "any" tool_choice ---
# With tool_choice="any" the model MUST call at least one tool, but it can choose which one.
# Even though the greeting doesn't logically require a tool, the model is forced to pick one.
openai_any_tool_choice_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key).bind_tools(tools, tool_choice="any")

any_tool_choice_conversation = [HumanMessage("Hello! I am a software developer. What about you?")]

# The model will pick one of the available tools even for this casual question
any_tool_choice_response = openai_any_tool_choice_model.invoke(any_tool_choice_conversation)
any_tool_choice_conversation.append(any_tool_choice_response)


In [ ]:
# Print the "any" tool_choice conversation — the response will contain a tool call
print_conversation(any_tool_choice_conversation)


## Forced tool choice

In [ ]:
# --- Forced tool_choice (specific tool name) ---
# Passing the exact tool name forces the model to ALWAYS call that specific tool,
# regardless of what the user says. This is useful when you need to guarantee
# a particular action happens every time (e.g., logging, auditing, or a required step).
openai_forced_tool_choice_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key).bind_tools(tools, tool_choice=do_nothing.name)

forced_tool_choice_conversation = [HumanMessage("Hello! I am a software developer. What about you?")]

# The model MUST call `do_nothing`, even though it makes no sense for this conversation
forced_tool_choice_response = openai_forced_tool_choice_model.invoke(forced_tool_choice_conversation)
forced_tool_choice_conversation.append(forced_tool_choice_response)


In [ ]:
# Print the forced tool_choice conversation — the response will always call do_nothing
print_conversation(forced_tool_choice_conversation)
